# Migration to GRIB2 - practical 3
## Vertical levels: `levtype=hl` and `levtype=sol`

In GRIB1, many fields that are really "a parameter at a given level" are encoded as separate **surface**
parameters: `100u` for the u-wind at 100 m, `stl1` ... `stl4` for the soil temperature in the four soil layers.

In GRIB2 the **parameter** and the **level** are kept apart: **one paramId for all levels**, with the level
given by `levelist`.

| Case | What changes | Parameters |
| --- | --- | --- |
| 1 | **nothing**: levels at 10 m and below stay in `levtype=sfc` | 2 m temperature, 10 m wind |
| 2 | heights **above 10 m** move to **`levtype=hl`** (height levels) | 100 m and 200 m wind |
| 3 | soil layers move to **`levtype=sol`** (surface other levels) | soil temperature, volumetric soil moisture |

All data come from **MARS**, for the same forecast, **1 September 2026 00 UTC**:  
- Mix of GRIB1 and GRIB2 (depending on the parameter) with `expver=1`, and
- the GRIB2 test data with **`expver=2250`**.

`expver` changes only because the GRIB2 data are a test dataset. Once GRIB2 is operational, it will be
`expver=1` again.

`grid` and `area` interpolate the data to 0.25° over Europe, to keep the downloads small.

## Setup

You need **earthkit-data** (with the mars source), **earthkit-plots** and a recent **ecCodes under the hood**.  
ecCodes older than 2.42.0 cannot decode the new encoding correctly - always use the latest release.  

If you are using this outside of ECMWF HPC, you will also need ecmwf-api-client which can be installed using pip.

In [1]:
import earthkit.data as ekd

# The keys we print for every field
keys = ["metadata.edition", "metadata.shortName", "metadata.paramId", "metadata.levtype",
        "metadata.typeOfLevel", "metadata.level", "metadata.levelist"]

## Levels at 10 m and below: no change

Parameters at 2 m and 10 m above ground - such as 2 m temperature, 2 m dewpoint, 10 m wind - are **exempt**.
They keep their paramId and stay in `levtype=sfc`, without `levelist`.

### GRIB1

In [2]:
near_sfc_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",                # changed
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",             # same
        "param": "167/165/166",       # same: 2t/10u/10v
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [3]:
near_sfc_grib1.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist
0,1,2t,167,sfc,surface,0,None
1,1,10u,165,sfc,surface,0,None
2,1,10v,166,sfc,surface,0,None


### GRIB2

In [4]:
near_sfc_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # changed (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",             # same
        "param": "167/165/166",       # same: 2t/10u/10v
        "timespan": "none",           # NEW: instantaneous fields
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [5]:
near_sfc_grib2.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist
0,2,2t,167,sfc,heightAboveGround,2,None
1,2,10u,165,sfc,heightAboveGround,10,None
2,2,10v,166,sfc,heightAboveGround,10,None


| | GRIB1 | GRIB2 | |
| --- | --- | --- | --- |
| `param` | 167/165/166 | 167/165/166 | same |
| `levtype` | `sfc` | `sfc` | same |
| `levelist` | - | - | same |
| level in the data | `surface`, 0 | `heightAboveGround`, 2 or 10 | changed |

The request stays the same. Only the encoding inside the file is more precise: GRIB2 says the field is 2 m or
10 m above ground.

## Heights above 10 m: `levtype=hl`

### GRIB1

Four paramIds, one per parameter and height.

In [ ]:
wind_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",                # changed
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",             # changed
        "param": "228246/228247/228239/228240",   # changed: 100u/100v/200u/200v
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [7]:
wind_grib1.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist
0,1,100u,228246,sfc,surface,0,None
1,1,100v,228247,sfc,surface,0,None
2,1,200u,228239,sfc,surface,0,None
3,1,200v,228240,sfc,surface,0,None


Note that the GRIB1 metadata says `surface`, level 0: the height (100 m, 200 m) is only known from the
paramId.

### GRIB2

Two paramIds - the ordinary `u` and `v` - and the height in `levelist`.

In [ ]:
wind_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # changed (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "hl",              # changed: height levels
        "levelist": "100/200",        # NEW: height above ground in m
        "param": "131/132",           # changed: u/v
        "timespan": "none",           # NEW: instantaneous fields
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [9]:
wind_grib2.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist
0,2,u,131,hl,heightAboveGround,100,100
1,2,v,132,hl,heightAboveGround,100,100
2,2,u,131,hl,heightAboveGround,200,200
3,2,v,132,hl,heightAboveGround,200,200


The values are the same:

In [ ]:
wind_grib1.ls(keys=["metadata.shortName", "metadata.paramId", "metadata.levelist", "metadata.maximum"])

In [11]:
wind_grib2.ls(keys=["metadata.shortName", "metadata.paramId", "metadata.levelist", "metadata.maximum"])

,metadata.shortName,metadata.paramId,metadata.levelist,metadata.maximum
0,u,131,100,25.440830
1,v,132,100,16.173437
2,u,131,200,28.333492
3,v,132,200,18.486898


| GRIB1 `param` (`levtype=sfc`) | GRIB2 `param` | GRIB2 `levtype` | GRIB2 `levelist` |
| --- | --- | --- | --- |
| 228246 (`100u`) | 131 (`u`) | `hl` | 100 |
| 228247 (`100v`) | 132 (`v`) | `hl` | 100 |
| 228239 (`200u`) | 131 (`u`) | `hl` | 200 |
| 228240 (`200v`) | 132 (`v`) | `hl` | 200 |
| 228249 (`100si`) | 10 (`ws`) | `hl` | 100 |
| 228241 (`200si`) | 10 (`ws`) | `hl` | 200 |

> **Watch out:** `u` at 100 m now has the same paramId (131) as `u` on pressure levels or model levels.
> Selecting by paramId or shortName alone is no longer enough - always use the **level type** and level too.

The old paramIds are **not** available in GRIB2. Asking for `100u` with `levtype=sfc` returns no data:

In [12]:
try:
    ekd.from_source(
        "mars",
        {
            "class": "od",
            "stream": "oper",
            "type": "fc",
            "expver": "2250",
            "date": "20260901",
            "time": "00",
            "levtype": "sfc",
            "param": "228246",        # old paramId: 100u
            "timespan": "none",
            "step": "24",
            "grid": "0.25/0.25",
            "area": "75/-30/25/45",
        },
    ).to_fieldlist()
except Exception as e:
    print("Request failed:", e)

2026-10-14 13:39:45 ECMWF API python library 1.7.0
2026-10-14 13:39:45 ECMWF API at https://api.ecmwf.int/v1
2026-10-14 13:39:45 Welcome Milana Vuckovic
2026-10-14 13:39:46 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2026-10-14 13:39:46 Request submitted
2026-10-14 13:39:46 Request id: 6acf7812b089271f8c8dbe1d
2026-10-14 13:39:46 Request is submitted
2026-10-14 13:39:47 Request is queued
2026-10-14 13:50:13 Request is active
2026-10-14 13:56:55 Calling 'nice mars /tmp/20261014-1240/c2/tmp-_mars-4fJe18-815e12a01a881070d6175e4c6c2a9bec.req'
2026-10-14 13:56:55 Forcing MIR_CACHE_PATH=/data/ec_coeff
2026-10-14 13:56:55 mars - WARN -
2026-10-14 13:56:55 mars - WARN -
2026-10-14 13:56:55 MIR environment variables:
2026-10-14 13:56:55 MIR_CACHE_PATH=/data/ec_coeff
2026-10-14 13:56:55 MIR_LSM_NAMED=1km.climate.v013
2026-10-14 13:56:55 Using MARS binary: /usr/local/apps/mars/versions/6.34.7.0/bin/mars.bin
2026-10-14 

## Soil layers: `levtype=sol`

### GRIB1

Eight paramIds: soil temperature and volumetric soil water, one per layer.

In [ ]:
soil_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",                # changed
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",             # changed
        "param": "139/170/183/236/39/40/41/42", # changed: stl1/stl2/stl3/stl4/swvl1/swvl2/swvl3/swvl4
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [14]:
soil_grib1.ls(keys=keys + ["metadata.maximum"])

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist,metadata.maximum
0,1,stl1,139,sfc,depthBelowLandLayer,0,None,313.047684
1,1,stl2,170,sfc,depthBelowLandLayer,7,None,317.576431
2,1,stl3,183,sfc,depthBelowLandLayer,28,None,324.311066
3,1,stl4,236,sfc,depthBelowLandLayer,100,None,316.756256
4,1,swvl1,39,sfc,depthBelowLandLayer,0,None,0.759506
5,1,swvl2,40,sfc,depthBelowLandLayer,7,None,0.748489
6,1,swvl3,41,sfc,depthBelowLandLayer,28,None,0.751221
7,1,swvl4,42,sfc,depthBelowLandLayer,100,None,0.752884


### GRIB2

Two paramIds - `sot` and `vsw` - and the layer in `levelist`.

In [ ]:
soil_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # changed (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sol",             # changed: surface other levels
        "levelist": "1/2/3/4",        # NEW: soil layer number
        "param": "260360/260199",     # changed: sot/vsw
        "timespan": "none",           # NEW: instantaneous fields
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [16]:
soil_grib2.ls(keys=keys + ["metadata.maximum"])

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist,metadata.maximum
0,2,sot,260360,sol,soilLayer,1,1,313.047684
1,2,vsw,260199,sol,soilLayer,1,1,0.759506
2,2,sot,260360,sol,soilLayer,2,2,317.576431
3,2,vsw,260199,sol,soilLayer,2,2,0.748489
4,2,sot,260360,sol,soilLayer,3,3,324.311066
5,2,vsw,260199,sol,soilLayer,3,3,0.751221
6,2,sot,260360,sol,soilLayer,4,4,316.756256
7,2,vsw,260199,sol,soilLayer,4,4,0.752884


| GRIB1 `param` (`levtype=sfc`) | GRIB2 `param` | GRIB2 `levtype` | GRIB2 `levelist` | layer depth |
| --- | --- | --- | --- | --- |
| 139 / 170 / 183 / 236 (`stl1` ... `stl4`) | 260360 (`sot`) | `sol` | 1 / 2 / 3 / 4 | |
| 39 / 40 / 41 / 42 (`swvl1` ... `swvl4`) | 260199 (`vsw`) | `sol` | 1 / 2 / 3 / 4 | |
| | | | 1 | 0-7 cm |
| | | | 2 | 7-28 cm |
| | | | 3 | 28-100 cm |
| | | | 4 | 100-289 cm |

Same values, but look at `level`: in GRIB1 it is the **top of the layer in cm** (0, 7, 28, 100), in GRIB2 it is
the **layer number** (1, 2, 3, 4). Code that uses `level` to find a soil layer needs to change.

## Your turn now

### Practical A
Modify the requests below to download the wind speed at 100 m and 200 m (`100si`, `200si`). (You can find the
paramIds [here](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-Parametersatheightsaboveground))

In [ ]:
ws_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",                # Add the levtype here
        "param": "228249/228241",     # Add paramIds here
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
ws_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "hl",                # Add the levtype here
        "levelist": "100/200",          # Add the heights here
        "param": "10",                  # Add paramIds here
        "timespan": "none",             # Add the timespan here
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [19]:
ws_grib1.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist
0,1,100si,228249,sfc,surface,0,None
1,1,200si,228241,sfc,surface,0,None


In [20]:
ws_grib2.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist
0,2,ws,10,hl,heightAboveGround,100,100
1,2,ws,10,hl,heightAboveGround,200,200


### Practical B
Modify the requests below to download the sea-ice temperature in the four ice layers (`istl1` ... `istl4`).
(You can find the paramIds [here](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-Soiltemperature,volumetricsoilmoistureandsea-icetemperature))

Compare the two. What is the `typeOfLevel` in GRIB2?

In [ ]:
ice_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",                # Add the levtype here
        "param": "35/36/37/38",          # Add paramIds here
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
ice_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sol",                # Add the levtype here
        "levelist": "1/2/3/4",           # Add the layers here
        "param": "262024",                  # Add paramIds here
        "timespan": "none",              # Add the timespan here
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [24]:
ice_grib1.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist
0,1,istl1,35,sfc,depthBelowLandLayer,0,None
1,1,istl2,36,sfc,depthBelowLandLayer,7,None
2,1,istl3,37,sfc,depthBelowLandLayer,28,None
3,1,istl4,38,sfc,depthBelowLandLayer,100,None


In [25]:
ice_grib2.ls(keys=keys)

,metadata.edition,metadata.shortName,metadata.paramId,metadata.levtype,metadata.typeOfLevel,metadata.level,metadata.levelist
0,2,sit,262024,sol,seaIceLayer,1,1
1,2,sit,262024,sol,seaIceLayer,2,2
2,2,sit,262024,sol,seaIceLayer,3,3
3,2,sit,262024,sol,seaIceLayer,4,4


Further reading:
- [Migration to GRIB2 - Test GRIB2 data in MARS](https://confluence.ecmwf.int/pages/viewpage.action?pageId=716086411)
- [Changes in the MARS language](https://confluence.ecmwf.int/pages/viewpage.action?pageId=621051006)
- [Migration to GRIB2 - changes to encoding of parameters](https://confluence.ecmwf.int/pages/viewpage.action?pageId=554148197)